In [ ]:
import mlflow
import dagshub
import joblib

In [2]:
# Dags Hub Variáveis
REPO_OWNER = "JosueJNLui"
REPO_NAME = "fiap-mlet-challenge-fase-1"

# Nome do experimento no MLflow
EXPERIMENT_NAME = "Telecon-Churn"

In [3]:
# Configuração de Rastreabilidade atrabés do MLflow + DagsHub
dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

Accessing as JosueJNLui

Initialized MLflow to track repo "JosueJNLui/fiap-mlet-challenge-fase-1"

Repository JosueJNLui/fiap-mlet-challenge-fase-1 initialized!

In [23]:
modelos = ["DummyClassifier_kfold_validation", "LogisticRegression_kfold_validation", "RandomForest_kfold_validation", 
            "MLP_lr=0.01_dropout=0.15_batch=128_hidden_dims=8",
            "MLP_lr=0.01_dropout=0.3_batch=64_hidden_dims=8",
            "MLP_lr=0.001_dropout=0.3_batch=32_hidden_dims=32"]

In [25]:
# Buscamos apenas apenas as Runs desejadas (top 3 MLPs + baselines)
df_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

df_final = df_runs.set_index('tags.mlflow.runName').loc[modelos]

# Selecionando apenas o essencial para a monografia
tabela_tese = df_final[[
    'metrics.kfold_lucro_medio', 
    'metrics.roc_auc', 
    'metrics.recall',
    'metrics.custo_falso_positivo_BRL'
]]

display(tabela_tese)

,metrics.kfold_lucro_medio,metrics.roc_auc,metrics.recall,metrics.custo_falso_positivo_BRL
tags.mlflow.runName,,,,
DummyClassifier_kfold_validation,-74750.0,0.500000,0.000000,0.0
LogisticRegression_kfold_validation,33120.0,0.849513,0.964523,21910.0
RandomForest_kfold_validation,32340.0,0.841290,0.957199,21700.0
MLP_lr=0.01_dropout=0.15_batch=128_hidden_dims=8,33120.0,0.847806,0.956515,20830.0
MLP_lr=0.01_dropout=0.3_batch=64_hidden_dims=8,32980.0,0.849074,0.955817,20880.0
MLP_lr=0.001_dropout=0.3_batch=32_hidden_dims=32,33100.0,0.846366,0.959893,21300.0


In [ ]:
#Embora modelos de diferentes complexidades tenham apresentado lucros líquidos convergentes, a arquitetura MLP demonstrou uma superioridade marginal na gestão de Falsos Positivos. Notavelmente, a configuração MLP com 8 neurônios atingiu o mesmo patamar de rentabilidade da Regressão Logística, porém reduzindo em aproximadamente 4,9% os gastos com ações de retenção desnecessárias. 

In [ ]:
# Hold-out Test

In [27]:
modelos = ["DummyClassifier", "LogisticRegression", "RandomForest", "MLP_GridSearch_KFold"]

In [32]:
# Buscamos apenas apenas as Runs desejadas (top 3 MLPs + baselines)
df_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

df_final = df_runs.set_index('tags.mlflow.runName').loc[modelos]

# Selecionando apenas o essencial para a monografia
tabela_tese = df_final[[
    'metrics.roc_auc', 
    'metrics.recall',
    'metrics.f1_score',
    'metrics.accuracy',
    'metrics.precision',
    'metrics.lucro_liquido_BRL', 
    'metrics.custo_churn_perdido_BRL',
    'metrics.custo_falso_positivo_BRL',
]]

display(tabela_tese)

,metrics.roc_auc,metrics.recall,metrics.f1_score,metrics.accuracy,metrics.precision,metrics.lucro_liquido_BRL,metrics.custo_churn_perdido_BRL,metrics.custo_falso_positivo_BRL
tags.mlflow.runName,,,,,,,,
DummyClassifier,0.500000,0.000000,0.000000,0.734564,0.000000,-187000.0,187000.0,0.0
LogisticRegression,0.620429,0.959893,0.434127,0.599716,0.395374,81200.0,7500.0,54900.0
RandomForest,0.715963,0.954545,0.490811,0.599716,0.394912,79600.0,8500.0,54700.0
MLP_GridSearch_KFold,0.846831,0.959893,0.559626,0.599006,0.394939,81100.0,7500.0,55000.0


In [ ]:
# Embora os modelos de Regressão Logística e MLP tenham apresentado resultados de lucro líquido convergentes
# no conjunto de teste (R$ 81.200,00 e R$ 81.100,00, respectivamente), a análise aprofundada das métricas de performance revela uma
# superioridade técnica determinante da arquitetura de redes neurais.

# A principal evidência reside na métrica ROC-AUC, onde a MLP atingiu a marca de 0.8468, superando significativamente a Regressão Logística, que
# obteve apenas 0.6204. Esta discrepância de aproximadamente 36% na capacidade de discriminação indica que, enquanto a Regressão Logística
# opera próxima ao limite da aleatoriedade para separar as classes, a MLP demonstra uma fronteira de decisão robusta e confiável.

# A relevância desta superioridade reflete-se em três pilares fundamentais:

# - Robustez ao Threshold: Modelos com AUC elevado são menos sensíveis a variações no limiar de decisão (threshold). Isso significa que a
#   MLP possui uma 'margem de segurança' estatística maior, garantindo que o lucro projetado não seja fruto de uma calibração fortuita, mas sim
#   de um aprendizado sólido sobre os padrões de churn.

# - Captura de Não-Linearidade: O salto de performance no AUC confirma que os fatores determinantes para o churn neste dataset não possuem relações
#   estritamente lineares. A MLP, através de suas camadas ocultas e funções de ativação, foi capaz de mapear interações complexas entre variáveis que
#    modelos tradicionais falharam em capturar.

# - Eficiência a Longo Prazo: Em um cenário de produção real, onde o comportamento do usuário é dinâmico, modelos com maior poder de separação
#   tendem a degradar mais lentamente (model drift). Assim, a escolha da MLP justifica-se não apenas pelo ganho imediato, mas pela estabilidade
#   e confiabilidade da solução em um ambiente de Big Data.

# Desta forma, conclui-se que a MLP_GridSearch_KFold é o modelo mais apto para implementação, oferecendo a melhor relação entre rentabilidade
# financeira e segurança estatística.